# Demo notebook cho preprocessing

## Thiết kế module tiền xử lý dữ liệu (Outlier + Datatype) theo kiến trúc Backend-based

### Mục tiêu chung
Xây dựng 2 processor (OutlierProcessor, DatatypeProcessor) có thể chạy thống nhất trên nhiều backend:
- **pandas** (CPU)
- **cuDF** (GPU)
- **numpy** (thông qua `NumpyFrame`)

Thiết kế nhấn mạnh:
- **Tách logic xử lý khỏi framework** (pandas/cuDF/numpy)
- **Round-trip I/O**: nhận nhiều kiểu input → coerce về “frame-like” → xử lý → trả lại đúng kiểu output ban đầu
- **Trace/debug rõ ràng**: mỗi bước lưu `last_transform_details_`

---

## i) Kiến trúc nền tảng

### `NumpyFrame`
Wrapper “DataFrame-like” cho numpy:
- Lưu cột dạng `dict[str, np.ndarray]` (mỗi cột 1D)
- Hỗ trợ tối thiểu: `columns`, `__len__`, `__getitem__`, `__setitem__`, `copy()`

Mục đích: giúp numpy có giao diện giống DataFrame để processor không phải viết riêng theo numpy.

### `BackendBase` (abstraction layer)
Định nghĩa các primitive ops mà processor cần:
- **Numeric ops**: `to_numeric`, `quantile`, `median`, `mean`, `clip`
- **Categorical ops**: `value_counts`, `isin`
- **Mask ops**: `sum_mask`, `any_mask`, `set_where`, `filter_rows_keep`
- **Datetime + casting**: `parse_datetime_series`, `cast_series`, `cast_frame_column`
- **I/O conversion**: `df_from_numpy_2d/structured`, `to_pandas`, `frame_to_numpy_2d`

=> Processor chỉ gọi backend ops, không phụ thuộc pandas/cuDF/numpy.

### Coercion layer
- `_coerce_input_to_frame(...)`: đưa input về đúng “frame-like” theo `prefer_backend/prefer_gpu`
- `_coerce_frame_to_output(...)`: trả output đúng kiểu như input (`pandas/cudf/numpy_structured/numpy_2d`)
- `BaseProcessor` giữ `input_kind_`, `input_schema_`, `feature_names_in_` để đảm bảo consistency giữa fit/transform.

---

## ii) Xử lý Outlier (OutlierProcessor)

### Input
- `numeric_cols`: các cột số → phát hiện outlier theo **IQR**
- `categorical_cols`: cột phân loại → phát hiện outlier theo **low-frequency** (ít xuất hiện)
- `OutlierConfig`: cấu hình chiến lược xử lý

### Output
- Dataset đã được làm sạch theo rule
- `last_transform_details_` gồm:
  - bounds IQR, số lượng outlier theo cột số
  - danh sách low-frequency categories theo cột category
  - số bản ghi bị drop (nếu chọn remove)

### Thiết kế nội bộ
#### a) `NumericOutlierProcessor`
- **fit**: tính `(q1, q3, iqr)` cho từng cột → lưu bounds `(lower, upper)`
- **transform**:
  - tạo `col_mask = (s < lower) | (s > upper)`
  - xử lý theo `numeric_strategy`:
    - `clip`: dùng `backend.clip(s, lower, upper)`
    - `replace`: dùng `backend.set_where(..., repl)` với repl = median/mean/constant
    - `remove`: chỉ đánh dấu mask, để OutlierProcessor xử lý drop cuối cùng

#### b) `CategoricalOutlierProcessor`
- **fit**: dùng `backend.value_counts()` để tìm nhóm `< cat_min_count`
- **transform**:
  - `col_mask = backend.isin(col, low_cats)`
  - xử lý theo `cat_strategy`:
    - `replace`: set label “Other”
    - `remove`: đánh dấu để drop cuối pipeline

#### c) `OutlierProcessor` (orchestration)
- chạy num → cat
- gom mask remove (nếu strategy = remove)
- drop hàng bằng `backend.filter_rows_keep()`
- ghi log chi tiết vào `last_transform_details_`

---

## iii) Chuẩn hóa kiểu dữ liệu (DatatypeProcessor)

### Input
- Dataset + `DatatypeConfig`
  - `timestamp_rules`: danh sách quy tắc parse datetime (format, unit, tz, drop_tz, errors)
  - `cast_map`: `{col -> target_dtype}`

### Output
- Dataset đã đúng schema dtype
- `last_transform_details_` gồm:
  - thay đổi dtype trước/sau cho timestamp
  - thay đổi dtype trước/sau cho cast_map + `na_count`

### Thiết kế luồng xử lý
1. **Validate columns**: kiểm tra thiếu cột dựa trên `cast_map` và `timestamp_rules` (strict thì raise)
2. **Apply timestamp rules** (ưu tiên chạy trước)
   - gọi `backend.parse_datetime_series(col, rule)`
3. **Apply cast_map**
   - gọi `backend.cast_frame_column(col, target, errors=...)`
   - log `before/after/na_count`

### Quy ước & xử lý đặc biệt theo backend
- `timestamp_rules` chịu trách nhiệm chính cho datetime (format + tz).
- `cast_map` là bước “final schema casting”.
- Với **cuDF** có hạn chế: `cudf.to_datetime(errors="coerce")` trên series không hỗ trợ.
  - Thiết kế giải quyết bằng:
    - parse datetime ở `timestamp_rules` (có thể fallback pandas khi cần tz)
    - trong `_apply_cast_map`: nếu backend là cudf và target là datetime
      - **skip** nếu cột đã là datetime
      - hoặc ép `errors="raise"` nếu cần cast lại (fail-fast)

---


## 0. Base class and backend

In [1]:
from __future__ import annotations

from abc import ABC, abstractmethod
from dataclasses import asdict, dataclass
from typing import Any, Dict, List, Optional, Tuple, Union, Literal

import numpy as np
import pandas as pd
import cudf


class NumpyFrame:
    """
    Minimal DataFrame-like wrapper for numpy.

    - Store columns as dict[str, np.ndarray] (1D arrays)
    - Support: columns, __len__, __getitem__, __setitem__, copy()
    - Row filtering and set-where are done by backend (not via .loc)
    """

    def __init__(self, data: Dict[str, np.ndarray], columns: List[str]) -> None:
        self._data = {c: np.asarray(data[c]) for c in columns}
        self._columns = list(columns)

        n = None
        for c in self._columns:
            if self._data[c].ndim != 1:
                raise ValueError(f"NumpyFrame expects 1D arrays per column, got {c} with shape {self._data[c].shape}")
            n = len(self._data[c]) if n is None else n
            if len(self._data[c]) != n:
                raise ValueError("All columns must have the same number of rows.")

    @property
    def columns(self) -> List[str]:
        return list(self._columns)

    def __len__(self) -> int:
        return len(self._data[self._columns[0]]) if self._columns else 0

    def __getitem__(self, col: str) -> np.ndarray:
        return self._data[col]

    def __setitem__(self, col: str, values: Any) -> None:
        if col not in self._data:
            raise KeyError(f"Column '{col}' not found in NumpyFrame.")
        arr = np.asarray(values)
        if arr.ndim != 1:
            raise ValueError(f"Assigned column must be 1D array, got shape {arr.shape}.")
        if len(arr) != len(self):
            raise ValueError(f"Assigned column length mismatch: got {len(arr)}, expected {len(self)}.")
        self._data[col] = arr

    def copy(self) -> "NumpyFrame":
        return NumpyFrame({c: self._data[c].copy() for c in self._columns}, self._columns)

    def to_dict(self) -> Dict[str, np.ndarray]:
        return {c: self._data[c] for c in self._columns}


class BackendBase(ABC):
    name: str

    # ---- frame io / schema ----
    @abstractmethod
    def is_frame(self, X: Any) -> bool: ...

    @abstractmethod
    def columns(self, X: Any) -> List[str]: ...

    @abstractmethod
    def df_from_numpy_structured(self, X: np.ndarray) -> Any: ...

    @abstractmethod
    def df_from_numpy_2d(self, X: np.ndarray, columns: List[str]) -> Any: ...

    @abstractmethod
    def to_pandas(self, X: Any) -> Any: ...

    @abstractmethod
    def frame_to_numpy_2d(self, X: Any, columns: List[str]) -> np.ndarray: ...

    @abstractmethod
    def to_numeric(self, s: Any, *, errors: str) -> Any: ...

    @abstractmethod
    def quantile(self, s: Any, q: float) -> float: ...

    @abstractmethod
    def median(self, s: Any) -> float: ...

    @abstractmethod
    def mean(self, s: Any) -> float: ...

    @abstractmethod
    def clip(self, s: Any, lower: float, upper: float) -> Any: ...

    @abstractmethod
    def isin(self, s: Any, values: List[Any]) -> Any: ...

    @abstractmethod
    def value_counts(self, s: Any, *, dropna: bool) -> Tuple[List[Any], List[int]]: ...

    @abstractmethod
    def sum_mask(self, mask: Any) -> int: ...

    @abstractmethod
    def any_mask(self, mask: Any) -> bool: ...

    @abstractmethod
    def set_where(self, X: Any, mask: Any, col: str, value: Any) -> None: ...

    @abstractmethod
    def filter_rows_keep(self, X: Any, keep_mask: Any) -> Any: ...

    @abstractmethod
    def parse_datetime_series(self, s: Any, rule: TimestampRule) -> Any: ...

    @abstractmethod
    def cast_series(self, s: Any, target: CastTarget, *, errors: ErrorsPolicy) -> Any: ...

    @abstractmethod
    def cast_frame_column(self, X: Any, col: str, target: CastTarget, *, errors: ErrorsPolicy) -> Dict[str, Any]: ...


class PandasBackend(BackendBase):
    name = "pandas"

    def __init__(self) -> None:
        import pandas as pd  # noqa
        self._pd = pd

    def is_frame(self, X: Any) -> bool:
        return type(X).__module__.startswith("pandas") and hasattr(X, "columns")

    def columns(self, X: Any) -> List[str]:
        return [str(c) for c in list(X.columns)]

    def df_from_numpy_structured(self, X: np.ndarray) -> Any:
        return self._pd.DataFrame.from_records(X)

    def df_from_numpy_2d(self, X: np.ndarray, columns: List[str]) -> Any:
        return self._pd.DataFrame(X, columns=columns)

    def to_pandas(self, X: Any) -> Any:
        return X

    def frame_to_numpy_2d(self, X: Any, columns: List[str]) -> np.ndarray:
        return X[columns].to_numpy()

    def to_numeric(self, s: Any, *, errors: str) -> Any:
        return self._pd.to_numeric(s, errors=errors)

    def quantile(self, s: Any, q: float) -> float:
        return float(s.quantile(q))

    def median(self, s: Any) -> float:
        return float(s.median())

    def mean(self, s: Any) -> float:
        return float(s.mean())

    def clip(self, s: Any, lower: float, upper: float) -> Any:
        return s.clip(lower, upper)

    def isin(self, s: Any, values: List[Any]) -> Any:
        return s.isin(values)

    def value_counts(self, s: Any, *, dropna: bool) -> Tuple[List[Any], List[int]]:
        vc = s.value_counts(dropna=dropna)
        return vc.index.tolist(), vc.values.tolist()

    def sum_mask(self, mask: Any) -> int:
        return int(mask.sum())

    def any_mask(self, mask: Any) -> bool:
        return bool(mask.any())

    def set_where(self, X: Any, mask: Any, col: str, value: Any) -> None:
        X.loc[mask, col] = value

    def filter_rows_keep(self, X: Any, keep_mask: Any) -> Any:
        return X.loc[keep_mask].copy()

    def parse_datetime_series(self, s: Any, rule: TimestampRule) -> Any:
        dt = self._pd.to_datetime(
            s,
            format=rule.fmt,
            unit=rule.unit,
            errors=rule.errors,
            utc=False,
        )

        # tz localize/convert (pandas robust)
        if rule.tz_in:
            try:
                dt = dt.dt.tz_localize(rule.tz_in, ambiguous="NaT", nonexistent="NaT")
            except Exception:
                pass

        if rule.tz_out:
            try:
                if getattr(dt.dt, "tz", None) is None:
                    dt = dt.dt.tz_localize("UTC", ambiguous="NaT", nonexistent="NaT")
                dt = dt.dt.tz_convert(rule.tz_out)
            except Exception:
                pass

        if rule.drop_tz:
            try:
                dt = dt.dt.tz_localize(None)
            except Exception:
                pass

        return dt

    def cast_series(self, s: Any, target: CastTarget, *, errors: ErrorsPolicy) -> Any:
        if target == "datetime64[ns]":
            return self._pd.to_datetime(s, errors=errors)

        if target == "float":
            return self._pd.to_numeric(s, errors=errors).astype(float)

        if target == "int":
            num = self._pd.to_numeric(s, errors=errors)
            return num.astype("Int64")

        if target == "string":
            return s.astype("string")

        if target == "category":
            return s.astype("category")

        if target == "bool":
            if errors == "ignore":
                return s
            ss = s.astype(str).str.strip().str.lower()
            true_set = ["1", "true", "t", "yes", "y"]
            false_set = ["0", "false", "f", "no", "n"]
            out = ss.isin(true_set)

            if errors == "coerce":
                known = ss.isin(true_set + false_set)
                out = out.astype("boolean")
                out = out.where(known, other=self._pd.NA)

            return out

        raise ValueError(f"Unsupported target: {target}")

    def cast_frame_column(self, X: Any, col: str, target: CastTarget, *, errors: ErrorsPolicy) -> Dict[str, Any]:
        before = str(getattr(X[col], "dtype", None))
        X[col] = self.cast_series(X[col], target, errors=errors)
        after = str(getattr(X[col], "dtype", None))

        na_count = 0
        try:
            na_count = int(self._pd.isna(X[col]).sum())
        except Exception:
            pass

        return {"col": col, "before": before, "after": after, "na_count": na_count, "target": target}


class CudfBackend(BackendBase):
    name = "cudf"

    def __init__(self) -> None:
        import cudf  # type: ignore
        self._cudf = cudf

    def is_frame(self, X: Any) -> bool:
        return type(X).__module__.startswith("cudf") and hasattr(X, "columns")

    def columns(self, X: Any) -> List[str]:
        return [str(c) for c in list(X.columns)]

    def df_from_numpy_structured(self, X: np.ndarray) -> Any:
        cols = list(X.dtype.names or [])
        return self._cudf.DataFrame({c: X[c] for c in cols})

    def df_from_numpy_2d(self, X: np.ndarray, columns: List[str]) -> Any:
        try:
            return self._cudf.DataFrame(X, columns=columns)
        except Exception:
            return self._cudf.DataFrame({col: X[:, i] for i, col in enumerate(columns)})

    def to_pandas(self, X: Any) -> Any:
        return X.to_pandas()

    def frame_to_numpy_2d(self, X: Any, columns: List[str]) -> np.ndarray:
        return X[columns].to_pandas().to_numpy()

    def to_numeric(self, s: Any, *, errors: str) -> Any:
        return self._cudf.to_numeric(s, errors=errors)

    def quantile(self, s: Any, q: float) -> float:
        return float(s.quantile(q))

    def median(self, s: Any) -> float:
        return float(s.median())

    def mean(self, s: Any) -> float:
        return float(s.mean())

    def clip(self, s: Any, lower: float, upper: float) -> Any:
        return s.clip(lower, upper)

    def isin(self, s: Any, values: List[Any]) -> Any:
        return s.isin(values)

    def value_counts(self, s: Any, *, dropna: bool) -> Tuple[List[Any], List[int]]:
        vc = s.value_counts(dropna=dropna)
        idx = vc.index.to_arrow().to_pylist() if hasattr(vc.index, "to_arrow") else list(vc.index)
        vals = vc.to_pandas().to_list() if hasattr(vc, "to_pandas") else list(vc.values)
        return idx, [int(v) for v in vals]

    def sum_mask(self, mask: Any) -> int:
        return int(mask.sum())

    def any_mask(self, mask: Any) -> bool:
        return bool(mask.any())

    def set_where(self, X: Any, mask: Any, col: str, value: Any) -> None:
        X.loc[mask, col] = value

    def filter_rows_keep(self, X: Any, keep_mask: Any) -> Any:
        return X.loc[keep_mask].copy()

    def parse_datetime_series(self, s: Any, rule: TimestampRule) -> Any:
        if rule.errors != "raise":
            raise ValueError(
                f"[CudfBackend] datetime parse only supports errors='raise'. "
                f"Got errors='{rule.errors}'. "
                f"Fix: set errors='raise' or run DatatypeProcessor with prefer_backend='pandas'."
            )
    
        if rule.tz_in or rule.tz_out:
            pdf = s.to_pandas() if hasattr(s, "to_pandas") else s
            pd_be = PandasBackend()
            out_pd = pd_be.parse_datetime_series(pdf, rule)
            return self._cudf.from_pandas(out_pd)
    
        # native cuDF parse (errors='raise' only)
        return self._cudf.to_datetime(
            s,
            format=rule.fmt,
            unit=rule.unit,
            errors="raise",
        )


    def cast_series(self, s: Any, target: CastTarget, *, errors: ErrorsPolicy) -> Any:
        if target == "datetime64[ns]":
            return self._cudf.to_datetime(s, errors=errors)

        if target == "float":
            return self._cudf.to_numeric(s, errors=errors).astype("float64")

        if target == "int":
            num = self._cudf.to_numeric(s, errors=errors)
            try:
                return num.astype("int64")
            except Exception:
                return num.astype("float64")

        if target == "string":
            return s.astype("str")

        if target == "category":
            return s.astype("category")

        if target == "bool":
            if errors == "ignore":
                return s
            ss = s.astype("str").str.strip().str.lower()
            return ss.isin(["1", "true", "t", "yes", "y"])

        raise ValueError(f"Unsupported target: {target}")

    def cast_frame_column(self, X: Any, col: str, target: CastTarget, *, errors: ErrorsPolicy) -> Dict[str, Any]:
        before = str(getattr(X[col], "dtype", None))
        X[col] = self.cast_series(X[col], target, errors=errors)
        after = str(getattr(X[col], "dtype", None))

        na_count = 0
        try:
            na_count = int(X[col].isna().sum())
        except Exception:
            pass

        return {"col": col, "before": before, "after": after, "na_count": na_count, "target": target}


class NumpyBackend(BackendBase):
    name = "numpy"

    def is_frame(self, X: Any) -> bool:
        return isinstance(X, NumpyFrame)

    def columns(self, X: Any) -> List[str]:
        return X.columns

    def df_from_numpy_structured(self, X: np.ndarray) -> Any:
        cols = list(X.dtype.names or [])
        return NumpyFrame({c: X[c] for c in cols}, cols)

    def df_from_numpy_2d(self, X: np.ndarray, columns: List[str]) -> Any:
        data = {col: np.asarray(X[:, i]) for i, col in enumerate(columns)}
        return NumpyFrame(data, columns)

    def to_pandas(self, X: Any) -> Any:
        import pandas as pd
        if isinstance(X, NumpyFrame):
            return pd.DataFrame({c: X[c] for c in X.columns})
        # fallback
        return pd.DataFrame(X)

    def frame_to_numpy_2d(self, X: Any, columns: List[str]) -> np.ndarray:
        if not isinstance(X, NumpyFrame):
            return self.to_pandas(X)[columns].to_numpy()
        return np.column_stack([np.asarray(X[c]) for c in columns])

    def to_numeric(self, s: Any, *, errors: str) -> Any:
        arr = np.asarray(s)
        if arr.dtype.kind in ("i", "u", "f"):
            return arr.astype(float, copy=False)
        import pandas as pd
        out = pd.to_numeric(arr, errors=errors).to_numpy()
        return out

    def quantile(self, s: Any, q: float) -> float:
        arr = np.asarray(s, dtype=float)
        return float(np.nanquantile(arr, q))

    def median(self, s: Any) -> float:
        arr = np.asarray(s, dtype=float)
        return float(np.nanmedian(arr))

    def mean(self, s: Any) -> float:
        arr = np.asarray(s, dtype=float)
        return float(np.nanmean(arr))

    def clip(self, s: Any, lower: float, upper: float) -> Any:
        arr = np.asarray(s, dtype=float)
        return np.clip(arr, lower, upper)

    def isin(self, s: Any, values: List[Any]) -> Any:
        return np.isin(np.asarray(s), np.asarray(values, dtype=object))

    def value_counts(self, s: Any, *, dropna: bool) -> Tuple[List[Any], List[int]]:
        arr = np.asarray(s, dtype=object)

        if dropna:
            # drop None / NaN
            mask = np.array([v is not None and not (isinstance(v, float) and np.isnan(v)) for v in arr], dtype=bool)
            arr = arr[mask]

        vals, counts = np.unique(arr, return_counts=True)
        return vals.tolist(), counts.astype(int).tolist()

    def sum_mask(self, mask: Any) -> int:
        return int(np.asarray(mask, dtype=bool).sum())

    def any_mask(self, mask: Any) -> bool:
        return bool(np.asarray(mask, dtype=bool).any())

    def set_where(self, X: Any, mask: Any, col: str, value: Any) -> None:
        if not isinstance(X, NumpyFrame):
            raise TypeError("NumpyBackend.set_where expects NumpyFrame")
        m = np.asarray(mask, dtype=bool)
        data = X[col].copy()
        data[m] = value
        X[col] = data

    def filter_rows_keep(self, X: Any, keep_mask: Any) -> Any:
        if not isinstance(X, NumpyFrame):
            raise TypeError("NumpyBackend.filter_rows_keep expects NumpyFrame")
        m = np.asarray(keep_mask, dtype=bool)
        return NumpyFrame({c: np.asarray(X[c])[m] for c in X.columns}, X.columns)

    def parse_datetime_series(self, s: Any, rule: TimestampRule) -> Any:
        import pandas as pd

        ser = pd.Series(np.asarray(s, dtype=object))
        dt = pd.to_datetime(
            ser,
            format=rule.fmt,
            unit=rule.unit,
            errors=rule.errors,
            utc=False,
        )

        if rule.tz_in:
            try:
                dt = dt.dt.tz_localize(rule.tz_in, ambiguous="NaT", nonexistent="NaT")
            except Exception:
                pass

        if rule.tz_out:
            try:
                if getattr(dt.dt, "tz", None) is None:
                    dt = dt.dt.tz_localize("UTC", ambiguous="NaT", nonexistent="NaT")
                dt = dt.dt.tz_convert(rule.tz_out)
            except Exception:
                pass

        if rule.drop_tz:
            try:
                dt = dt.dt.tz_localize(None)
            except Exception:
                pass

        return dt.to_numpy(dtype="datetime64[ns]")

    def cast_series(self, s: Any, target: CastTarget, *, errors: ErrorsPolicy) -> Any:
        import pandas as pd

        arr = np.asarray(s, dtype=object)
        ser = pd.Series(arr)

        if target == "datetime64[ns]":
            return pd.to_datetime(ser, errors=errors).to_numpy(dtype="datetime64[ns]")

        if target == "float":
            return pd.to_numeric(ser, errors=errors).astype(float).to_numpy()

        if target == "int":
            num = pd.to_numeric(ser, errors=errors)
            out = num.where(~num.isna(), other=pd.NA).astype("Int64")
            return out.astype(object).to_numpy()

        if target == "string":
            return ser.astype("string").astype(object).to_numpy()

        if target == "category":
            return ser.astype(object).to_numpy()

        if target == "bool":
            if errors == "ignore":
                return arr
            ss = ser.astype(str).str.strip().str.lower()
            true_set = ["1", "true", "t", "yes", "y"]
            false_set = ["0", "false", "f", "no", "n"]
            out = ss.isin(true_set)

            if errors == "coerce":
                known = ss.isin(true_set + false_set)
                out = out.astype("boolean").where(known, other=pd.NA)

            return out.astype(object).to_numpy()

        raise ValueError(f"Unsupported target: {target}")

    def cast_frame_column(self, X: Any, col: str, target: CastTarget, *, errors: ErrorsPolicy) -> Dict[str, Any]:
        before = str(getattr(np.asarray(X[col]).dtype, "name", np.asarray(X[col]).dtype))
        X[col] = self.cast_series(X[col], target, errors=errors)
        after = str(getattr(np.asarray(X[col]).dtype, "name", np.asarray(X[col]).dtype))

        na_count = 0
        try:
            import pandas as pd
            na_count = int(pd.isna(np.asarray(X[col], dtype=object)).sum())
        except Exception:
            pass

        return {"col": col, "before": before, "after": after, "na_count": na_count, "target": target}


def _try_get_cudf_backend() -> Optional[CudfBackend]:
    try:
        return CudfBackend()
    except Exception:
        return None


def _get_backend_for_frame(Xf: Any) -> BackendBase:
    if isinstance(Xf, NumpyFrame):
        return NumpyBackend()
    if type(Xf).__module__.startswith("cudf"):
        return CudfBackend()
    return PandasBackend()


def _is_numpy_structured(X: Any) -> bool:
    return isinstance(X, np.ndarray) and (X.dtype.names is not None)


def _is_numpy_2d(X: Any) -> bool:
    return isinstance(X, np.ndarray) and (X.dtype.names is None) and (X.ndim == 2)


PreferBackend = Literal["auto", "pandas", "cudf", "numpy"]


def _choose_backend_for_numpy(prefer_backend: PreferBackend, prefer_gpu: bool) -> BackendBase:
    if prefer_backend == "numpy":
        return NumpyBackend()
    if prefer_backend == "pandas":
        return PandasBackend()
    if prefer_backend == "cudf":
        cudf_be = _try_get_cudf_backend()
        if cudf_be is None:
            raise ImportError("prefer_backend='cudf' but cudf is not available.")
        return cudf_be
    # auto
    if prefer_gpu:
        cudf_be = _try_get_cudf_backend()
        if cudf_be is not None:
            return cudf_be
    return PandasBackend()


def _coerce_input_to_frame(
    X: Any,
    *,
    reset: bool,
    fitted_schema: Optional[Dict[str, Any]] = None,
    prefer_gpu: bool = True,
    prefer_backend: PreferBackend = "auto",
) -> Tuple[Any, str, Dict[str, Any]]:
    """
    Returns:
      - frame: pandas.DataFrame or cudf.DataFrame or NumpyFrame
      - kind:  "pandas"|"cudf"|"numpy_structured"|"numpy_2d"
      - schema: used for round-trip output conversion
    """
    # already frames
    pd_be = PandasBackend()
    if pd_be.is_frame(X):
        schema = {"backend": "pandas", "columns": pd_be.columns(X)}
        return X, "pandas", schema

    cudf_be = _try_get_cudf_backend()
    if cudf_be and cudf_be.is_frame(X):
        schema = {"backend": "cudf", "columns": cudf_be.columns(X)}
        return X, "cudf", schema

    be = _choose_backend_for_numpy(prefer_backend, prefer_gpu)

    # numpy structured
    if _is_numpy_structured(X):
        cols = list(X.dtype.names or [])
        frame = be.df_from_numpy_structured(X)
        schema = {"backend": be.name, "dtype": X.dtype, "columns": cols}
        return frame, "numpy_structured", schema

    # numpy 2d
    if _is_numpy_2d(X):
        if (not reset) and fitted_schema and "columns" in fitted_schema:
            cols = list(fitted_schema["columns"])
        else:
            cols = [f"x{i}" for i in range(X.shape[1])]
        frame = be.df_from_numpy_2d(X, cols)
        schema = {"backend": be.name, "columns": cols}
        return frame, "numpy_2d", schema

    raise TypeError(
        f"Unsupported input type: {type(X)}. "
        "Supported: pandas.DataFrame, cudf.DataFrame, numpy structured array, numpy 2D array."
    )


def _coerce_frame_to_output(frame: Any, *, kind: str, schema: Dict[str, Any]) -> Any:
    if kind in ("pandas", "cudf"):
        return frame

    be = _get_backend_for_frame(frame)

    if kind == "numpy_structured":
        # giữ dtype ban đầu (structured) ổn định
        dtype = schema["dtype"]
        pdf = be.to_pandas(frame)
        rec = pdf.to_records(index=False)
        return np.asarray(rec).astype(dtype, copy=False)

    if kind == "numpy_2d":
        cols = list(schema["columns"])
        return be.frame_to_numpy_2d(frame, cols)

    raise TypeError(f"Unsupported output kind: {kind}")

## 0.5. Validation object type (must be structure data)

In [2]:
class TransformerMixin:
    def fit_transform(self, X: Any, y: Any = None, **fit_params: Any) -> Any:
        if y is None:
            return self.fit(X, **fit_params).transform(X)
        return self.fit(X, y=y, **fit_params).transform(X)


class ParamMixin:
    def get_params(self, deep: bool = True) -> Dict[str, Any]:
        return {k: v for k, v in self.__dict__.items() if not k.endswith("_")}

    def set_params(self, **params: Any) -> "ParamMixin":
        for k, v in params.items():
            setattr(self, k, v)
        return self


class ValidationMixin:
    strict: bool = True
    required_cols: List[str]
    n_features_in_: int
    feature_names_in_: List[str]

    def _validate_frame(self, Xf: Any, *, reset: bool, backend: BackendBase) -> Any:
        if not hasattr(Xf, "columns") or not hasattr(Xf, "__len__") or not hasattr(Xf, "__getitem__"):
            raise TypeError(f"Expected frame-like object after coercion, got {type(Xf)}")

        feature_names = backend.columns(Xf)
        n_features = len(feature_names)

        if reset:
            self.n_features_in_ = n_features
            self.feature_names_in_ = feature_names
        else:
            if not hasattr(self, "n_features_in_"):
                raise RuntimeError("Not fitted: missing n_features_in_. Call fit() first.")
            if n_features != self.n_features_in_:
                raise ValueError(f"n_features mismatch: got {n_features}, expected {self.n_features_in_}.")
            if feature_names != getattr(self, "feature_names_in_", feature_names):
                raise ValueError(
                    "Feature names mismatch between fit and transform.\n"
                    f"- fit: {self.feature_names_in_}\n"
                    f"- now: {feature_names}"
                )

        if getattr(self, "required_cols", None):
            missing = [c for c in self.required_cols if c not in feature_names]
            if missing and self.strict:
                raise ValueError(f"Missing required columns: {missing}. Available: {feature_names}")

        return Xf

## Base Processer

In [3]:
class BaseProcessor(ParamMixin, TransformerMixin, ValidationMixin, ABC):
    name: str = "base_processor"
    version: str = "0.1"

    def __init__(
        self,
        *,
        required_cols: Optional[List[str]] = None,
        strict: bool = True,
        prefer_gpu: bool = True,
        prefer_backend: PreferBackend = "auto",
    ) -> None:
        self.required_cols = required_cols or []
        self.strict = strict
        self.prefer_gpu = prefer_gpu
        self.prefer_backend = prefer_backend

        self.is_fitted_: bool = False
        self.input_kind_: Optional[str] = None
        self.input_schema_: Dict[str, Any] = {}
        self.backend_: Optional[BackendBase] = None

    def fit(self, X: Any, y: Any = None) -> "BaseProcessor":
        Xf, kind, schema = _coerce_input_to_frame(
            X, reset=True, prefer_gpu=self.prefer_gpu, prefer_backend=self.prefer_backend
        )
        be = _get_backend_for_frame(Xf)
        Xf = self._validate_frame(Xf, reset=True, backend=be)

        self.input_kind_ = kind
        self.input_schema_ = schema
        self.backend_ = be

        self._fit(Xf, y=y)
        self.is_fitted_ = True
        return self

    def transform(self, X: Any) -> Any:
        if not self.is_fitted_:
            raise RuntimeError("Not fitted. Call fit() first.")

        Xf, kind, schema = _coerce_input_to_frame(
            X,
            reset=False,
            fitted_schema=self.input_schema_,
            prefer_gpu=self.prefer_gpu,
            prefer_backend=self.prefer_backend,
        )

        if self.input_kind_ is not None and kind != self.input_kind_:
            raise ValueError(f"Input kind mismatch: fit on '{self.input_kind_}', got '{kind}' at transform.")

        be = _get_backend_for_frame(Xf)
        Xf = self._validate_frame(Xf, reset=False, backend=be)

        out_frame = self._transform(Xf)
        return _coerce_frame_to_output(out_frame, kind=kind, schema=schema)

    @abstractmethod
    def _fit(self, X: Any, y: Any = None) -> None: ...

    @abstractmethod
    def _transform(self, X: Any) -> Any: ...

## 1. Outlier preprocessing

### a. Base class and utils

In [4]:
def _false_mask_like_frame(X: Any) -> Any:
    # tạo mask False theo backend (numpy/pandas/cudf đều OK với biểu thức so sánh)
    first_col = list(X.columns)[0]
    m = X[first_col] == "__never__"
    try:
        return m.fillna(False)  # pandas/cudf
    except Exception:
        return m  # numpy bool array


def _is_na(v: Any) -> bool:
    try:
        return (v is None) or (v != v)
    except Exception:
        return False

In [5]:
@dataclass
class OutlierConfig:
    iqr_k: float = 1.5
    cat_min_count: int = 2

    numeric_strategy: str = "clip"  # "remove" | "replace" | "clip"
    numeric_replace: Union[str, float, int] = "median"

    cat_strategy: str = "replace"  # "remove" | "replace"
    cat_replace_label: str = "Other"

    allow_zero_iqr: bool = True

In [6]:
class NumericOutlierProcessor:
    def __init__(self, cfg: OutlierConfig, backend: BackendBase) -> None:
        self.cfg = cfg
        self.be = backend
        self.numeric_bounds_: Dict[str, Tuple[float, float]] = {}

    def fit(self, X: Any, numeric_cols: List[str]) -> "NumericOutlierProcessor":
        bounds: Dict[str, Tuple[float, float]] = {}
        for col in numeric_cols:
            s = self.be.to_numeric(X[col], errors="coerce")
            q1 = self.be.quantile(s, 0.25)
            q3 = self.be.quantile(s, 0.75)
            iqr = q3 - q1

            if (not self.cfg.allow_zero_iqr) and float(iqr) == 0.0:
                lower, upper = float(q1), float(q3)
            else:
                lower = float(q1 - self.cfg.iqr_k * iqr)
                upper = float(q3 + self.cfg.iqr_k * iqr)

            bounds[col] = (lower, upper)

        self.numeric_bounds_ = bounds
        return self

    def transform(self, X: Any, numeric_cols: List[str]) -> Tuple[Any, Any, Dict[str, Any]]:
        X_out = X
        outlier_mask = None
        details: Dict[str, Any] = {}

        for col in numeric_cols:
            lower, upper = self.numeric_bounds_[col]
            s = self.be.to_numeric(X_out[col], errors="coerce")

            col_mask = (s < lower) | (s > upper)
            outlier_mask = col_mask if outlier_mask is None else (outlier_mask | col_mask)

            details[col] = {"lower": lower, "upper": upper, "outlier_count": self.be.sum_mask(col_mask)}

            if self.cfg.numeric_strategy == "clip":
                X_out[col] = self.be.clip(s, lower, upper)

            elif self.cfg.numeric_strategy == "replace":
                if self.cfg.numeric_replace == "median":
                    repl = self.be.median(s)
                elif self.cfg.numeric_replace == "mean":
                    repl = self.be.mean(s)
                else:
                    repl = float(self.cfg.numeric_replace)
                self.be.set_where(X_out, col_mask, col, repl)

            elif self.cfg.numeric_strategy == "remove":
                pass
            else:
                raise ValueError(f"Unsupported numeric_strategy: {self.cfg.numeric_strategy}")

        if outlier_mask is None:
            outlier_mask = _false_mask_like_frame(X_out)

        return X_out, outlier_mask, details


class CategoricalOutlierProcessor:
    def __init__(self, cfg: OutlierConfig, backend: BackendBase) -> None:
        self.cfg = cfg
        self.be = backend
        self.low_freq_categories_: Dict[str, List[Any]] = {}

    def fit(self, X: Any, categorical_cols: List[str]) -> "CategoricalOutlierProcessor":
        low_freq: Dict[str, List[Any]] = {}
        for col in categorical_cols:
            values, counts = self.be.value_counts(X[col], dropna=False)
            low_cats = [v for v, c in zip(values, counts) if c < self.cfg.cat_min_count]
            low_freq[col] = low_cats

        self.low_freq_categories_ = low_freq
        return self

    def transform(self, X: Any, categorical_cols: List[str]) -> Tuple[Any, Any, Dict[str, Any]]:
        X_out = X
        outlier_mask = None
        details: Dict[str, Any] = {}

        for col in categorical_cols:
            low_cats = list(self.low_freq_categories_.get(col, []))
            col_mask = self.be.isin(X_out[col], low_cats)
            outlier_mask = col_mask if outlier_mask is None else (outlier_mask | col_mask)

            details[col] = {
                "low_frequency_categories": [None if _is_na(v) else v for v in low_cats],
                "low_freq_count": self.be.sum_mask(col_mask),
                "min_count": self.cfg.cat_min_count,
            }

            if self.cfg.cat_strategy == "replace":
                self.be.set_where(X_out, col_mask, col, self.cfg.cat_replace_label)
            elif self.cfg.cat_strategy == "remove":
                pass
            else:
                raise ValueError(f"Unsupported cat_strategy: {self.cfg.cat_strategy}")

        if outlier_mask is None:
            outlier_mask = _false_mask_like_frame(X_out)

        return X_out, outlier_mask, details


class OutlierProcessor(BaseProcessor):
    name = "outlier_processor"
    version = "0.1"

    def __init__(
        self,
        *,
        cfg: Optional[Union[OutlierConfig, Dict[str, Any]]] = None,
        numeric_cols: Optional[List[str]] = None,
        categorical_cols: Optional[List[str]] = None,
        required_cols: Optional[List[str]] = None,
        strict: bool = True,
        prefer_gpu: bool = True,
        prefer_backend: PreferBackend = "auto",
    ) -> None:
        super().__init__(
            required_cols=required_cols,
            strict=strict,
            prefer_gpu=prefer_gpu,
            prefer_backend=prefer_backend,
        )

        if cfg is None:
            self.cfg = OutlierConfig()
        elif isinstance(cfg, OutlierConfig):
            self.cfg = cfg
        elif isinstance(cfg, dict):
            self.cfg = OutlierConfig(**cfg)
        else:
            raise TypeError(f"Unsupported cfg type: {type(cfg)}")

        self.numeric_cols = numeric_cols
        self.categorical_cols = categorical_cols

        self._num_proc: Optional[NumericOutlierProcessor] = None
        self._cat_proc: Optional[CategoricalOutlierProcessor] = None

    def _fit(self, X: Any, y: Any = None) -> None:
        if self.backend_ is None:
            raise RuntimeError("backend_ is not set.")

        numeric_cols = self.numeric_cols or []
        categorical_cols = self.categorical_cols or []

        self.numeric_cols_ = list(numeric_cols)
        self.categorical_cols_ = list(categorical_cols)

        self._num_proc = NumericOutlierProcessor(self.cfg, backend=self.backend_)
        self._cat_proc = CategoricalOutlierProcessor(self.cfg, backend=self.backend_)

        self._num_proc.fit(X, self.numeric_cols_)
        self._cat_proc.fit(X, self.categorical_cols_)

        self.numeric_bounds_ = dict(self._num_proc.numeric_bounds_)
        self.low_freq_categories_ = dict(self._cat_proc.low_freq_categories_)

    def _transform(self, X: Any) -> Any:
        if self.backend_ is None or self._num_proc is None or self._cat_proc is None:
            raise RuntimeError("Not fitted properly.")

        X_out = X.copy()

        X_out, num_mask, num_details = self._num_proc.transform(X_out, self.numeric_cols_)
        X_out, cat_mask, cat_details = self._cat_proc.transform(X_out, self.categorical_cols_)

        remove_mask = None
        if self.cfg.numeric_strategy == "remove":
            remove_mask = num_mask if remove_mask is None else (remove_mask | num_mask)
        if self.cfg.cat_strategy == "remove":
            remove_mask = cat_mask if remove_mask is None else (remove_mask | cat_mask)

        dropped = 0
        if remove_mask is not None and self.backend_.any_mask(remove_mask):
            before = len(X_out)
            keep = ~remove_mask
            X_out = self.backend_.filter_rows_keep(X_out, keep)
            dropped = int(before - len(X_out))

        self.last_transform_details_ = {
            "config": asdict(self.cfg),
            "numeric": num_details,
            "categorical": cat_details,
            "dropped_rows": dropped,
            "input_backend": getattr(self, "input_schema_", {}).get("backend"),
            "input_kind": getattr(self, "input_kind_", None),
        }
        return X_out


### b. Example

In [7]:
data = {
    "id": list(range(1, 11)),
    "price": [100, 120, 110, 130, 115, 125, 10000, 118, 122, 119],
    "quantity": [1, 2, 1, 2, 1, 2, 1, 1, 2, 1],
    "city": ["Hanoi", "Hanoi", "HCM", "HCM", "Danang", "Hanoi", "Hanoi", "Hue", "Hanoi", "HCM"],
}

cfg = OutlierConfig(
    iqr_k=1.5,
    cat_min_count=2,
    numeric_strategy="clip",   # "clip" / "replace" / "remove"
    cat_strategy="replace",    # "replace" / "remove"
    cat_replace_label="Other",
)

#### Pandas plugin

In [8]:
df_pd = pd.DataFrame(data)

proc_pd = OutlierProcessor(
    cfg=cfg,
    numeric_cols=["price", "quantity"],
    categorical_cols=["city"],
    prefer_gpu=False,
)

df_pd_out = proc_pd.fit_transform(df_pd)

print("=== pandas input ===")
print(df_pd)

print("\n=== pandas output ===")
print(df_pd_out)

print("\n=== pandas details ===")
print(proc_pd.last_transform_details_)

df_pd_new = pd.DataFrame(
    {
        "id": [11, 12, 13],
        "price": [121, 9999, 117],  # 9999 will be clipped
        "quantity": [1, 2, 1],
        "city": ["Hanoi", "Hue", "HCM"],  # Hue low-freq -> Other
    }
)

df_pd_new_out = proc_pd.transform(df_pd_new)

print("\n=== pandas transform input ===")
print(df_pd_new)

print("\n=== pandas transform output ===")
print(df_pd_new_out)

print("\n=== pandas details metadata ===")
print(proc_pd.last_transform_details_)

=== pandas input ===
   id  price  quantity    city
0   1    100         1   Hanoi
1   2    120         2   Hanoi
2   3    110         1     HCM
3   4    130         2     HCM
4   5    115         1  Danang
5   6    125         2   Hanoi
6   7  10000         1   Hanoi
7   8    118         1     Hue
8   9    122         2   Hanoi
9  10    119         1     HCM

=== pandas output ===
   id  price  quantity   city
0   1    103         1  Hanoi
1   2    120         2  Hanoi
2   3    110         1    HCM
3   4    130         2    HCM
4   5    115         1  Other
5   6    125         2  Hanoi
6   7    137         1  Hanoi
7   8    118         1  Other
8   9    122         2  Hanoi
9  10    119         1    HCM

=== pandas details ===
{'config': {'iqr_k': 1.5, 'cat_min_count': 2, 'numeric_strategy': 'clip', 'numeric_replace': 'median', 'cat_strategy': 'replace', 'cat_replace_label': 'Other', 'allow_zero_iqr': True}, 'numeric': {'price': {'lower': 103.0, 'upper': 137.0, 'outlier_count': 2}, '

#### Cudf plugin

In [9]:
import cudf  # type: ignore

df_cu = cudf.DataFrame(data)

proc_cu = OutlierProcessor(
    cfg=cfg,
    numeric_cols=["price", "quantity"],
    categorical_cols=["city"],
    prefer_gpu=True,
)

df_cu_out = proc_cu.fit_transform(df_cu)

print("\n\n=== cudf input ===")
print(df_cu)

print("\n=== cudf output ===")
print(df_cu_out)

print("\n=== cudf details ===")
print(proc_cu.last_transform_details_)

df_cu_new = cudf.DataFrame(
    {
        "id": [11, 12, 13],
        "price": [121, 9999, 117],
        "quantity": [1, 2, 1],
        "city": ["Hanoi", "Hue", "HCM"],
    }
)

df_cu_new_out = proc_cu.transform(df_cu_new)

print("\n=== cudf transform input ===")
print(df_cu_new)

print("\n=== cudf transform output ===")
print(df_cu_new_out)

print("\n=== cudf details metadata ===")
print(proc_cu.last_transform_details_)



=== cudf input ===
   id  price  quantity    city
0   1    100         1   Hanoi
1   2    120         2   Hanoi
2   3    110         1     HCM
3   4    130         2     HCM
4   5    115         1  Danang
5   6    125         2   Hanoi
6   7  10000         1   Hanoi
7   8    118         1     Hue
8   9    122         2   Hanoi
9  10    119         1     HCM

=== cudf output ===
   id  price  quantity   city
0   1    103         1  Hanoi
1   2    120         2  Hanoi
2   3    110         1    HCM
3   4    130         2    HCM
4   5    115         1  Other
5   6    125         2  Hanoi
6   7    137         1  Hanoi
7   8    118         1  Other
8   9    122         2  Hanoi
9  10    119         1    HCM

=== cudf details ===
{'config': {'iqr_k': 1.5, 'cat_min_count': 2, 'numeric_strategy': 'clip', 'numeric_replace': 'median', 'cat_strategy': 'replace', 'cat_replace_label': 'Other', 'allow_zero_iqr': True}, 'numeric': {'price': {'lower': 103.0, 'upper': 137.0, 'outlier_count': 2}, 'quan

#### Numpy plugin (Array and 2D)

In [10]:
X_np_struct = np.array(
    list(
        zip(
            data["id"],
            data["price"],
            data["quantity"],
            data["city"],
        )
    ),
    dtype=[("id", "i4"), ("price", "f8"), ("quantity", "i4"), ("city", "U16")],
)

proc_np = OutlierProcessor(
    cfg=cfg,
    numeric_cols=["price", "quantity"],
    categorical_cols=["city"],
    prefer_gpu=True,  # numpy --> cudf
)

X_np_out = proc_np.fit_transform(X_np_struct)

print("\n\n=== numpy structured input ===")
print(X_np_struct[:5])

print("\n=== numpy structured output ===")
print(X_np_out[:5])

print("\n=== numpy structured details ===")
print(proc_np.last_transform_details_)

X_np_new = np.array(
    [
        (11, 121.0, 1, "Hanoi"),
        (12, 9999.0, 2, "Hue"),
        (13, 117.0, 1, "HCM"),
    ],
    dtype=X_np_struct.dtype,
)

X_np_new_out = proc_np.transform(X_np_new)

print("\n=== numpy structured transform input ===")
print(X_np_new)

print("\n=== numpy structured transform output ===")
print(X_np_new_out)

print("\n=== numpy structured details metadata ===")
print(proc_np.last_transform_details_)



=== numpy structured input ===
[(1, 100., 1, 'Hanoi') (2, 120., 2, 'Hanoi') (3, 110., 1, 'HCM')
 (4, 130., 2, 'HCM') (5, 115., 1, 'Danang')]

=== numpy structured output ===
[(1, 103., 1, 'Hanoi') (2, 120., 2, 'Hanoi') (3, 110., 1, 'HCM')
 (4, 130., 2, 'HCM') (5, 115., 1, 'Other')]

=== numpy structured details ===
{'config': {'iqr_k': 1.5, 'cat_min_count': 2, 'numeric_strategy': 'clip', 'numeric_replace': 'median', 'cat_strategy': 'replace', 'cat_replace_label': 'Other', 'allow_zero_iqr': True}, 'numeric': {'price': {'lower': 103.0, 'upper': 137.0, 'outlier_count': 2}, 'quantity': {'lower': -0.5, 'upper': 3.5, 'outlier_count': 0}}, 'categorical': {'city': {'low_frequency_categories': ['Danang', 'Hue'], 'low_freq_count': 2, 'min_count': 2}}, 'dropped_rows': 0, 'input_backend': 'cudf', 'input_kind': 'numpy_structured'}

=== numpy structured transform input ===
[(11,  121., 1, 'Hanoi') (12, 9999., 2, 'Hue') (13,  117., 1, 'HCM')]

=== numpy structured transform output ===
[(11, 121., 1

In [11]:
X_np_2d = np.array(
    [
        [100.0, 1.0],
        [120.0, 2.0],
        [110.0, 1.0],
        [10000.0, 1.0],
    ],
    dtype=float,
)

proc_np2d = OutlierProcessor(
    cfg=OutlierConfig(iqr_k=1.5, numeric_strategy="clip", cat_strategy="replace"),
    numeric_cols=["x0", "x1"],
    categorical_cols=[],
    prefer_gpu=True,
)

X_np_2d_out = proc_np2d.fit_transform(X_np_2d)

print("\n\n=== numpy 2D input ===")
print(X_np_2d)

print("\n=== numpy 2D output ===")
print(X_np_2d_out)

print("\n=== numpy 2D details ===")
print(proc_np2d.last_transform_details_)



=== numpy 2D input ===
[[1.0e+02 1.0e+00]
 [1.2e+02 2.0e+00]
 [1.1e+02 1.0e+00]
 [1.0e+04 1.0e+00]]

=== numpy 2D output ===
[[1.00000e+02 1.00000e+00]
 [1.20000e+02 1.62500e+00]
 [1.10000e+02 1.00000e+00]
 [6.31375e+03 1.00000e+00]]

=== numpy 2D details ===
{'config': {'iqr_k': 1.5, 'cat_min_count': 2, 'numeric_strategy': 'clip', 'numeric_replace': 'median', 'cat_strategy': 'replace', 'cat_replace_label': 'Other', 'allow_zero_iqr': True}, 'numeric': {'x0': {'lower': -3616.25, 'upper': 6313.75, 'outlier_count': 1}, 'x1': {'lower': 0.625, 'upper': 1.625, 'outlier_count': 1}}, 'categorical': {}, 'dropped_rows': 0, 'input_backend': 'cudf', 'input_kind': 'numpy_2d'}


## 2. Datatype preprocessing

In [12]:
CastTarget = Literal[
    "int",
    "float",
    "string",
    "bool",
    "category",
    "datetime64[ns]",
]
ErrorsPolicy = Literal["raise", "coerce", "ignore"]


@dataclass
class TimestampRule:
    columns: List[str]
    fmt: Optional[str] = None
    unit: Optional[str] = None
    errors: ErrorsPolicy = "coerce"
    tz_in: Optional[str] = None
    tz_out: Optional[str] = None
    drop_tz: bool = True  # default: convert to naive datetime64[ns]


@dataclass
class DatatypeConfig:
    cast_map: Dict[str, CastTarget]
    timestamp_rules: List[TimestampRule] = None

In [13]:
class DatatypeProcessor(BaseProcessor):
    name = "datatype_processor"
    version = "0.1"

    def __init__(
        self,
        *,
        cfg: Optional[Union[DatatypeConfig, Dict[str, Any]]] = None,
        required_cols: Optional[List[str]] = None,
        strict: bool = True,
        prefer_gpu: bool = True,
        prefer_backend: PreferBackend = "auto",
    ) -> None:
        super().__init__(
            required_cols=required_cols,
            strict=strict,
            prefer_gpu=prefer_gpu,
            prefer_backend=prefer_backend,
        )

        if cfg is None:
            self.cfg = DatatypeConfig(cast_map={}, timestamp_rules=[])
        elif isinstance(cfg, DatatypeConfig):
            self.cfg = cfg
        elif isinstance(cfg, dict):
            ts_rules = cfg.get("timestamp_rules", []) or []
            # accept list[dict] too
            parsed_rules: List[TimestampRule] = []
            for r in ts_rules:
                if isinstance(r, TimestampRule):
                    parsed_rules.append(r)
                else:
                    parsed_rules.append(TimestampRule(**r))
            self.cfg = DatatypeConfig(
                cast_map=cfg.get("cast_map", {}) or {},
                timestamp_rules=parsed_rules,
            )
        else:
            raise TypeError(f"Unsupported cfg type: {type(cfg)}")

        if self.cfg.timestamp_rules is None:
            self.cfg.timestamp_rules = []

    def _fit(self, X: Any, y: Any = None) -> None:
        if self.backend_ is None:
            raise RuntimeError("backend_ is not set.")

        cols = set(self.backend_.columns(X))
        missing: List[str] = []

        for c in self.cfg.cast_map.keys():
            if c not in cols:
                missing.append(c)
        for r in self.cfg.timestamp_rules:
            for c in r.columns:
                if c not in cols:
                    missing.append(c)

        if missing and self.strict:
            raise ValueError(f"DatatypeProcessor: missing columns: {sorted(set(missing))}")

        self.cast_map_ = dict(self.cfg.cast_map)
        self.timestamp_rules_ = [asdict(r) for r in self.cfg.timestamp_rules]

    def _apply_timestamp_rules(self, X_out: Any) -> Dict[str, Any]:
        assert self.backend_ is not None

        details: Dict[str, Any] = {}
        for rule in self.cfg.timestamp_rules:
            for col in rule.columns:
                before = str(getattr(X_out[col], "dtype", None))
                X_out[col] = self.backend_.parse_datetime_series(X_out[col], rule)
                after = str(getattr(X_out[col], "dtype", None))
                details[col] = {"before": before, "after": after, "rule": asdict(rule)}
        return details

    def _apply_cast_map(self, X_out: Any) -> Dict[str, Any]:
        assert self.backend_ is not None
        cast_details: Dict[str, Any] = {}
    
        be_name = getattr(self.backend_, "name", "")
        for col, target in self.cfg.cast_map.items():
            if be_name == "cudf" and target == "datetime64[ns]":
                before = str(getattr(X_out[col], "dtype", None))
                if "datetime64" in before:
                    cast_details[col] = {
                        "col": col,
                        "before": before,
                        "after": before,
                        "na_count": int(X_out[col].isna().sum()) if hasattr(X_out[col], "isna") else None,
                        "target": target,
                        "note": "skip datetime cast on cudf (already parsed by timestamp_rules)",
                    }
                    continue
    
                cast_details[col] = self.backend_.cast_frame_column(
                    X_out, col, target, errors="raise"
                )
                continue
    
            cast_details[col] = self.backend_.cast_frame_column(
                X_out, col, target, errors="coerce"
            )
    
        return cast_details


    def _transform(self, X: Any) -> Any:
        if self.backend_ is None:
            raise RuntimeError("backend_ is not set.")

        X_out = X.copy()

        ts_details = self._apply_timestamp_rules(X_out)
        cast_details = self._apply_cast_map(X_out)

        self.last_transform_details_ = {
            "config": {
                "cast_map": dict(self.cfg.cast_map),
                "timestamp_rules": [asdict(r) for r in self.cfg.timestamp_rules],
            },
            "details": {
                "timestamp": ts_details,
                "cast": cast_details,
            },
            "input_backend": getattr(self, "input_schema_", {}).get("backend"),
            "input_kind": getattr(self, "input_kind_", None),
        }
        return X_out

### Example

#### Pandas plugin

In [14]:
df = pd.DataFrame(
    {
        "user_id": ["001", "002", "003", "004"],
        "age": ["18", "19", "bad", None],
        "score": ["1.5", "2.0", "3.2", "oops"],
        "event_time": ["2025-12-25 10:00:00", "2025-12-25 11:30:00", "bad_time", None],
        "city": ["HN", "HCM", "HN", "DN"],
        "is_active": ["true", "FALSE", "x", None],
    }
)

cfg = {
    "timestamp_rules": [
        {
            "columns": ["event_time"],
            "fmt": "%Y-%m-%d %H:%M:%S",
            "errors": "coerce",
            "tz_in": "Asia/Bangkok",
            "tz_out": "UTC",
            "drop_tz": True,
        }
    ],
    "cast_map": {
        "user_id": "string",
        "age": "int",
        "score": "float",
        "city": "category",
        "is_active": "bool",
        "event_time": "datetime64[ns]",
    },
}

proc = DatatypeProcessor(cfg=cfg, prefer_backend="auto", prefer_gpu=False)
df2 = proc.fit_transform(df)

print("=== BEFORE ===")
print(df.dtypes)
print(df)

print("\n=== AFTER ===")
print(df2.dtypes)
print(df2)

print("\n=== Details metadata ===")
print(proc.last_transform_details_)

=== BEFORE ===
user_id       object
age           object
score         object
event_time    object
city          object
is_active     object
dtype: object
  user_id   age score           event_time city is_active
0     001    18   1.5  2025-12-25 10:00:00   HN      true
1     002    19   2.0  2025-12-25 11:30:00  HCM     FALSE
2     003   bad   3.2             bad_time   HN         x
3     004  None  oops                 None   DN      None

=== AFTER ===
user_id       string[python]
age                    Int64
score                float64
event_time    datetime64[ns]
city                category
is_active            boolean
dtype: object
  user_id   age  score          event_time city  is_active
0     001    18    1.5 2025-12-25 03:00:00   HN       True
1     002    19    2.0 2025-12-25 04:30:00  HCM      False
2     003  <NA>    3.2                 NaT   HN       <NA>
3     004  <NA>    NaN                 NaT   DN       <NA>

=== Details metadata ===
{'config': {'cast_map': {'user_

#### Numpy plugin

In [15]:
X_np = np.array(
    [
        ["18", "1.5", "2025-12-25 10:00:00", "HN"],
        ["19", "2.0", "2025-12-25 11:30:00", "HCM"],
        ["bad", "3.2", "bad_time", "HN"],
        [None, "oops", None, "DN"],
    ],
    dtype=object,
)

cfg_np = {
    "timestamp_rules": [
        {
            "columns": ["x2"],
            "fmt": "%Y-%m-%d %H:%M:%S",
            "errors": "coerce",
            "drop_tz": True,
        }
    ],
    "cast_map": {
        "x0": "int",  # age
        "x1": "float",  # score
        "x2": "datetime64[ns]",  # event_time
        "x3": "category",  # city
    },
}

proc_np = DatatypeProcessor(cfg=cfg_np, prefer_backend="numpy", prefer_gpu=False)
X2 = proc_np.fit_transform(X_np)

print("\n=== BEFORE ===")
print(X_np)

print("\n=== AFTER ===")
print(X2)

print("\n=== Details metadata ===")
print(proc_np.last_transform_details_)



=== BEFORE ===
[['18' '1.5' '2025-12-25 10:00:00' 'HN']
 ['19' '2.0' '2025-12-25 11:30:00' 'HCM']
 ['bad' '3.2' 'bad_time' 'HN']
 [None 'oops' None 'DN']]

=== AFTER ===
[[18 1.5 1766656800000000000 'HN']
 [19 2.0 1766662200000000000 'HCM']
 [<NA> 3.2 None 'HN']
 [<NA> nan None 'DN']]

=== Details metadata ===
{'config': {'cast_map': {'x0': 'int', 'x1': 'float', 'x2': 'datetime64[ns]', 'x3': 'category'}, 'timestamp_rules': [{'columns': ['x2'], 'fmt': '%Y-%m-%d %H:%M:%S', 'unit': None, 'errors': 'coerce', 'tz_in': None, 'tz_out': None, 'drop_tz': True}]}, 'details': {'timestamp': {'x2': {'before': 'object', 'after': 'datetime64[ns]', 'rule': {'columns': ['x2'], 'fmt': '%Y-%m-%d %H:%M:%S', 'unit': None, 'errors': 'coerce', 'tz_in': None, 'tz_out': None, 'drop_tz': True}}}, 'cast': {'x0': {'col': 'x0', 'before': 'object', 'after': 'object', 'na_count': 2, 'target': 'int'}, 'x1': {'col': 'x1', 'before': 'object', 'after': 'float64', 'na_count': 1, 'target': 'float'}, 'x2': {'col': 'x2', '

#### Cudf plugin

In [16]:
df_gpu = cudf.DataFrame(
    {
        "user_id": ["001", "002", "003", "004"],
        "age": ["18", "19", "20", None],
        "score": ["1.5", "2.0", "3.2", "oops"],
        "event_time": ["2025-12-25 10:00:00", "2025-12-25 11:30:00", "2025-12-25 12:00:00", None],
        "city": ["HN", "HCM", "HN", "DN"],
        "is_active": ["true", "FALSE", "x", None],
    }
)

cfg = {
    "timestamp_rules": [
        {
            "columns": ["event_time"],
            "fmt": "%Y-%m-%d %H:%M:%S",
            "errors": "raise", 
            "tz_in": "Asia/Bangkok",
            "tz_out": "UTC",
            "drop_tz": True,
        }
    ],
    "cast_map": {
        "user_id": "string",
        "age": "int",
        "score": "float",
        "city": "category",
        "is_active": "bool",
        "event_time": "datetime64[ns]",
    },
}

proc_gpu = DatatypeProcessor(cfg=cfg, prefer_backend="cudf", prefer_gpu=True)

df2_gpu = proc_gpu.fit_transform(df_gpu)
print(df2_gpu.dtypes)
print(df2_gpu)

user_id               object
age                    int64
score                float64
event_time    datetime64[ns]
city                category
is_active               bool
dtype: object
  user_id   age score                     event_time city  is_active
0     001    18   1.5  2025-12-25 03:00:00.000000000   HN       True
1     002    19   2.0  2025-12-25 04:30:00.000000000  HCM      False
2     003    20   3.2  2025-12-25 05:00:00.000000000   HN      False
3     004  <NA>  <NA>                            NaT   DN      False
